In [0]:
from pyspark.sql import functions as F
from datetime import datetime,timezone



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.landing;
CREATE VOLUME IF NOT EXISTS workspace.landing.customer_files;

In [0]:
SOURCE_TABLE = "workspace.erp.customer"
LANDING_PATH = (
    "/Volumes/workspace/landing/customer_files/customer_cdc"
)

CHECKPOINT_PATH = (
    "/Volumes/workspace/landing/customer_files/"
    "_checkpoints/customer_cdf_to_landing"
)

In [0]:
batch_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

cdc_df = (
    spark.readStream
    .option("readChangeFeed","true")
    .table(SOURCE_TABLE)
)

In [0]:
landing_cdc_df = (
    cdc_df \
    .withColumn("_op",
        F.when(F.col("_change_type") == "update_preimage", "UPDATE_BEFORE")
        .when(F.col("_change_type") == "update_postimage", "UPDATE_AFTER")
        .when(F.col("_change_type") == "insert", "INSERT")
        .when(F.col("_change_type") == "delete", "DELETE")
        .otherwise(F.lit("UNKNOWN"))
    ) \
    .withColumn("_event_timestamp",F.col("_commit_timestamp"))
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn("_source_system",F.lit("ERP"))
    .withColumn("_landing_ingested_at",F.current_timestamp())
)

In [0]:
query = (
    landing_cdc_df.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation",CHECKPOINT_PATH) \
    .partitionBy("_batch_id") \
    .trigger(availableNow=True) \
    .start(LANDING_PATH)
)

query.awaitTermination()

In [0]:

landing_df = (spark.read.format("parquet").load(LANDING_PATH))


In [0]:
display(landing_df.groupBy("_batch_id","_change_type","_op").count() \
    .orderBy("_batch_id","_change_type"))

In [0]:
display(
    landing_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "_change_type",
        "_op",
        "_commit_version",
        "_commit_timestamp",
        "_batch_id"
    )
    .orderBy(
        "_commit_version",
        "customer_id",
        "_change_type"
    )
)

In [0]:
display(
    spark.read
    .format("parquet")
    .load(LANDING_PATH)
    .filter(
        F.col("_change_type").isin(
            "update_preimage",
            "update_postimage",
            "delete"
        )
        | (F.col("customer_id") == 17)
    )
    .select(
        "customer_id",
        "email",
        "total_purchases",
        "_change_type",
        "_op",
        "_commit_version",
        "_commit_timestamp",
        "_batch_id"
    )
    .orderBy(
        "_commit_version",
        "_change_type"
    )
)